In [29]:
from modeling_module.utils.date_util import DateUtil
import polars as pl

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'


target_raw = (pl.read_parquet(MAC_DIR + 'parquets/dyn_demand.parquet')
                .select(['oper_part_no', 'demand_dt', 'demand_qty'])
                .with_columns(pl.col('demand_dt').cast(pl.Utf8).str.to_date(format='%Y%m%d').alias('demand_dt'))
              )
target_raw_weekly = (target_raw
                    .with_columns(pl.col('demand_dt').map_elements(DateUtil.date_to_yyyyww, return_dtype = pl.Int64).alias('demand_weekly'))
                    .select(['oper_part_no', 'demand_weekly', 'demand_qty'])
                    .group_by(['oper_part_no', 'demand_weekly'])
                    .agg(pl.col('demand_qty').sum().alias('demand_qty'))
)
target_raw_monthly = (target_raw
                      .with_columns(pl.col('demand_dt').map_elements(DateUtil.date_to_yyyymm, return_dtype = pl.Int64).alias('demand_monthly'))
                      .select(['oper_part_no', 'demand_monthly', 'demand_qty'])
                      .group_by(['oper_part_no', 'demand_monthly'])
                      .agg(pl.col('demand_qty').sum().alias('demand_qty'))
                      )


In [31]:
target_raw_weekly = (target_raw_weekly
                        # .rename({'oper_part_no': 'unique_id', 'demand_weekly': 'ds', 'demand_qty': 'y'})
                        .with_columns(pl.col('demand_weekly').map_elements(DateUtil.yyyyww_to_date, return_dtype = pl.Date))
                     )
target_raw_weekly

oper_part_no,demand_weekly,demand_qty
str,date,f64
"""K5200-A0205BB""",2023-05-15,10.0
"""T2185-38041""",2019-05-27,5.0
"""05012-01363""",2019-06-03,5.0
"""05122-50640""",2019-04-01,3.0
"""T4145-82131""",2022-10-10,4.0
…,…,…
"""T4931-47091""",2021-11-01,1.0
"""T4620-44592""",2023-11-13,3.0
"""E5500-21332""",2022-04-04,5.0


In [35]:
from statsforecast import StatsForecast
from statsforecast.models import CrostonOptimized, CrostonSBA, CrostonClassic

season_length = 52
horizon = 27

models = [CrostonSBA(), CrostonOptimized(), CrostonClassic()]
sf = StatsForecast(models = models, freq = '1w')

# sf.fit(df = target_raw_weekly)
result = sf.fit_predict(h = 27, df = target_raw_weekly, i_col = 'oper_part_no', time_col = 'demand_weekly', target_col = 'demand_qty')
result



oper_part_no,demand_weekly,CrostonSBA,CrostonOptimized,CrostonClassic
str,date,f64,f64,f64
"""0001-1001""",2023-09-04,10.45311,11.00341,11.003273
"""0001-1001""",2023-09-11,10.45311,11.00341,11.003273
"""0001-1001""",2023-09-18,10.45311,11.00341,11.003273
"""0001-1001""",2023-09-25,10.45311,11.00341,11.003273
"""0001-1001""",2023-10-02,10.45311,11.00341,11.003273
…,…,…,…,…
"""ZZ90239""",2023-11-27,0.95,1.0,1.0
"""ZZ90239""",2023-12-04,0.95,1.0,1.0
"""ZZ90239""",2023-12-11,0.95,1.0,1.0


oper_part_no,demand_weekly,CrostonOptimized
str,date,f64
"""0001-1001""",2023-09-04,11.00341
"""0001-1001""",2023-09-11,11.00341
"""0001-1001""",2023-09-18,11.00341
"""0001-1001""",2023-09-25,11.00341
"""0001-1001""",2023-10-02,11.00341
…,…,…
"""ZZ90239""",2023-11-27,1.0
"""ZZ90239""",2023-12-04,1.0
"""ZZ90239""",2023-12-11,1.0
